# Entity and article entity representations

BT4222 Group 6 · representations 2/3. Run `3_01_news_embeddings.ipynb` first.

Validate the **training-release 100D entity coordinate system**, pool matched entity vectors into
article vectors, and distinguish missing annotations from missing lookup vectors. Relation embeddings
are excluded because the supplied news metadata does not include the corresponding graph triples.

Run top to bottom. Shared algorithms, validation and file handling live in `src/newsrec/`; this notebook controls settings and shows results. You do not run those Python files separately. Saved artifacts and `REBUILD` work as before.

In [ ]:
from pathlib import Path
import os
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display

# Find the source package whether this notebook starts in the root or its subfolder.
CODE_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents,
                            Path('/content/drive/MyDrive/BT4222'))
                  if (p / 'src/newsrec').is_dir()), None)
if CODE_ROOT is None:
    raise FileNotFoundError('Open this notebook from the project checkout containing src/newsrec/.')
sys.path.insert(0, str(CODE_ROOT / 'src'))
from newsrec.common import Settings, saved_stage_ready

DATA_ROOT = None  # Optional: point to another folder holding the project's saved datasets.
ROOT = Path(DATA_ROOT or os.environ.get('NEWSREC_ROOT') or CODE_ROOT).expanduser().resolve()
ARTIFACTS = ROOT / '3_Representations/artifacts'
print('Data/output root:', ROOT)

from newsrec.representations import build_entities

In [ ]:
# Defaults live in src/newsrec/common.py; override with Settings(...) if needed.
SETTINGS = Settings()
REBUILD = False  # Reuse saved outputs; True regenerates this notebook's results.

## 1. Parse and pool

Parse JSON; deduplicate Wikidata IDs across title and abstract; normalise each matched entity vector,
mean-pool, then normalise the article vector. Empty/unmatched articles use zero plus a validity mask.
The dev lookup is audited for coordinate differences, never merged automatically into the train space.
Zero vectors, malformed rows, nonfinite coordinates and duplicate entity IDs are checked explicitly.

In [ ]:
if REBUILD or not saved_stage_ready(ROOT, SETTINGS, "entity"):
    coverage = build_entities(ROOT, SETTINGS)
else:
    print("Loaded saved entity representations.")
    report = pd.read_csv(ROOT / "results/representation_coverage.csv")
    coverage = report[report.stage.eq("entity")]
display(coverage)

## 2. Audit coverage at the right denominator

Entity-ID lookup coverage, article-vector coverage and user-profile coverage are different quantities.
The preparation report's test lookup coverage uses the dev file; this notebook reports the canonical
train lookup for every split. It separately records whether the dev coordinates match.

In [ ]:
quality = pd.read_parquet(ARTIFACTS / "entity_quality.parquet")
display(quality.head())
display(
    quality.groupby(["title_entity_status", "abstract_entity_status"])
    .size()
    .rename("articles")
    .reset_index()
)
meta = json.loads((ARTIFACTS / "entity_manifest.json").read_text())
display(meta["dev_compatibility"])
vectors = np.load(ARTIFACTS / "article_entity100.npy", mmap_mode="r")
assert vectors.shape == (len(quality) + 1, SETTINGS.entity_dim)
assert np.isfinite(vectors).all() and np.all(vectors[0] == 0)
assert np.array_equal(
    np.linalg.norm(vectors[1:], axis=1) > 0, quality.has_entity_vector.to_numpy()
)
print("Article entity matrix MB:", vectors.nbytes / 1e6)

## Handoff

`article_entity100.npy` shares the article-row map with the text vectors. Entity and text coordinates
remain separate. `article_entity_links.parquet` supplies annotation links (including unmatched IDs)
for later graph construction; these annotation links are not evidence that an embedding exists.
The following notebook produces supplied-only entity user profiles for the clustering owner.